# Fake Review Detection - Model Training

In [12]:
# Import all required packages

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import os


In [3]:
# Load train and test datasets from the Data folder

data_dir = 'Data'

df_train = pd.read_csv(os.path.join(data_dir, 'train.csv'), sep=';')
df_test = pd.read_csv(os.path.join(data_dir, 'test.csv'), sep=';')

print(f'Train: {len(df_train)}, Test: {len(df_test)}')

Train: 7000, Test: 1500


## Prepare Features

In [4]:
# Separate raw text (features) from labels

X_train = df_train['text']
y_train = df_train['ai_generated']

X_test = df_test['text']
y_test = df_test['ai_generated']

## TF-IDF Vectorization

In [5]:
# Convert raw review text into TF-IDF feature vectors (fit on train only)

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## Hyperparameter Tuning + Logistic Regression

In [6]:
# Tune Logistic Regression hyperparameters using 5-fold cross-validation

from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': ['balanced']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_tfidf, y_train)
model = grid.best_estimator_
print(f'Best params: {grid.best_params_}')
print(f'Best CV F1:  {grid.best_score_:.4f}')
print(f'Vocabulary size: {len(tfidf.get_feature_names_out())}')

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best params: {'C': 10, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'liblinear'}
Best CV F1:  0.9269
Vocabulary size: 10000


c:\Users\behna\miniconda3\envs\fake-review\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


## Evaluation on Train Set

In [7]:
# Evaluate the tuned model on the training set

y_pred_train = model.predict(X_train_tfidf)

print('=== Train Set ===')
print(f'Accuracy:  {accuracy_score(y_train, y_pred_train):.4f}')
print(f'F1-Score:  {f1_score(y_train, y_pred_train):.4f}')
print(f'Precision: {precision_score(y_train, y_pred_train):.4f}')
print(f'Recall:    {recall_score(y_train, y_pred_train):.4f}')

=== Train Set ===
Accuracy:  0.9929
F1-Score:  0.9929
Precision: 0.9934
Recall:    0.9923


## Evaluation on Test Set

In [8]:
# Evaluate the tuned model on the held-out test set

y_pred = model.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)

report = classification_report(y_test, y_pred, output_dict=True)

print('=== Test Set ===')
print(f'Accuracy:  {acc:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred))
print(f'Human Prec: {report["0"]["precision"]:.4f}')
print(f'Human Rec:  {report["0"]["recall"]:.4f}')
print(f'Human F1:   {report["0"]["f1-score"]:.4f}')
print(f'AI Prec:    {report["1"]["precision"]:.4f}')
print(f'AI Rec:     {report["1"]["recall"]:.4f}')
print(f'AI F1:      {report["1"]["f1-score"]:.4f}')
print()
from sklearn.metrics import roc_auc_score
y_prob = model.predict_proba(X_test_tfidf)[:, 1]
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}')

=== Test Set ===
Accuracy:  0.9207
F1-Score:  0.9201
Precision: 0.9269
Recall:    0.9133

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.93      0.92       750
           1       0.93      0.91      0.92       750

    accuracy                           0.92      1500
   macro avg       0.92      0.92      0.92      1500
weighted avg       0.92      0.92      0.92      1500

Human Prec: 0.9146
Human Rec:  0.9280
Human F1:   0.9212
AI Prec:    0.9269
AI Rec:     0.9133
AI F1:      0.9201

ROC-AUC:   0.9774


## Top 10 Markers for AI (Fake) and Human (Real)

In [9]:
# Show the top 10 most predictive words/n-grams for each class

feature_names = np.array(tfidf.get_feature_names_out())
coefs = model.coef_[0]
top_ai_idx = np.argsort(coefs)[-10:][::-1]
top_human_idx = np.argsort(coefs)[:10]

print("=== Top 10 Markers for AI-Generated (Fake) ===")
for i in top_ai_idx:
    print(f"  {feature_names[i]:25s} weight: {coefs[i]:.4f}")

print("\n=== Top 10 Markers for Human-Written (Real) ===")
for i in top_human_idx:
    print(f"  {feature_names[i]:25s} weight: {coefs[i]:.4f}")

=== Top 10 Markers for AI-Generated (Fake) ===
  bit                       weight: 7.7611
  and the                   weight: 6.6970
  ve                        weight: 6.6947
  incredibly                weight: 5.3169
  but the                   weight: 5.1200
  immediately               weight: 4.9775
  actually                  weight: 4.8424
  finally                   weight: 4.8001
  completely                weight: 4.6706
  exactly                   weight: 4.5969

=== Top 10 Markers for Human-Written (Real) ===
  not                       weight: -7.3583
  product                   weight: -7.0896
  good                      weight: -6.3501
  nice                      weight: -6.1180
  to                        weight: -5.6461
  would                     weight: -5.5900
  will                      weight: -5.5713
  cute                      weight: -5.4616
  this is                   weight: -5.2432
  it is                     weight: -5.1479


## Leave-One-Category-Out Evaluation

In [10]:
# Leave-One-Category-Out (LOCO) cross-domain evaluation
# For each product category: train on all other categories, test on the held-out one.
# This tests whether the model generalizes to product types it has never seen.

from sklearn.metrics import precision_recall_fscore_support

results_loc = []
categories = sorted(df_train['category'].unique())

for held_out in categories:
    # Split: train on everything except the held-out category
    train_sub = df_train[df_train['category'] != held_out]
    test_sub = df_test[df_test['category'] == held_out]

    # Vectorize text with a fresh TF-IDF (fit only on this training subset)
    tfidf_loc = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
    X_tr = tfidf_loc.fit_transform(train_sub['text'])
    X_te = tfidf_loc.transform(test_sub['text'])

    # Train Logistic Regression with the best hyperparameters from tuning
    m = LogisticRegression(C=10, class_weight='balanced', solver='liblinear', max_iter=1000, random_state=42)
    m.fit(X_tr, train_sub['ai_generated'])
    pred = m.predict(X_te)

    # Store metrics for this category
    report = classification_report(test_sub['ai_generated'], pred, output_dict=True, zero_division=0)
    results_loc.append({
        'category': held_out,
        'accuracy': accuracy_score(test_sub['ai_generated'], pred),
        'precision': precision_score(test_sub['ai_generated'], pred, zero_division=0),
        'recall': recall_score(test_sub['ai_generated'], pred, zero_division=0),
        'f1': f1_score(test_sub['ai_generated'], pred, zero_division=0),
        'precision_human': report['0']['precision'],
        'recall_human': report['0']['recall'],
        'f1_human': report['0']['f1-score'],
        'precision_ai': report['1']['precision'],
        'recall_ai': report['1']['recall'],
        'f1_ai': report['1']['f1-score'],
        'macro_f1': report['macro avg']['f1-score'],
        'samples': len(test_sub)
    })

# Build results table and add an AVERAGE row
loc_df = pd.DataFrame(results_loc)
avg_row = pd.DataFrame([{
    'category': 'AVERAGE',
    'accuracy': loc_df['accuracy'].mean(),
    'precision': loc_df['precision'].mean(),
    'recall': loc_df['recall'].mean(),
    'f1': loc_df['f1'].mean(),
    'samples': loc_df['samples'].sum()
}])
pd.concat([loc_df, avg_row], ignore_index=True)

,category,accuracy,precision,recall,f1,precision_human,recall_human,f1_human,precision_ai,recall_ai,f1_ai,macro_f1,samples
0,Automotive,0.898734,0.958333,0.766667,0.851852,0.872727,0.979592,0.923077,0.958333,0.766667,0.851852,0.887464,79
1,Beauty_and_Personal_Care,0.910891,0.933333,0.875000,0.903226,0.892857,0.943396,0.917431,0.933333,0.875000,0.903226,0.910328,101
2,Books,0.948718,1.000000,0.913043,0.954545,0.888889,1.000000,0.941176,1.000000,0.913043,0.954545,0.947861,78
3,Cell_Phones_and_Accessories,0.846154,0.767442,0.846154,0.804878,0.901639,0.846154,0.873016,0.767442,0.846154,0.804878,0.838947,104
4,Clothing_Shoes_and_Jewelry,0.804054,0.950413,0.688623,0.798611,0.702857,0.953488,0.809211,0.950413,0.688623,0.798611,0.803911,296
5,Electronics,0.938967,0.943925,0.935185,0.939535,0.933962,0.942857,0.938389,0.943925,0.935185,0.939535,0.938962,213
6,Health_and_Household,0.909091,0.911111,0.854167,0.881720,0.907895,0.945205,0.926174,0.911111,0.854167,0.881720,0.903947,121
7,Home_and_Kitchen,0.865385,0.902256,0.805369,0.851064,0.837989,0.920245,0.877193,0.902256,0.805369,0.851064,0.864128,312
8,Kindle_Store,0.872093,0.933333,0.888889,0.910569,0.730769,0.826087,0.775510,0.933333,0.888889,0.910569,0.843040,86
9,Tools_and_Home_Improvement,0.909091,0.903846,0.903846,0.903846,0.913793,0.913793,0.913793,0.903846,0.903846,0.903846,0.908820,110


In [11]:
short_names = {
    'Automotive': 'Automotive',
    'Beauty_and_Personal_Care': 'Beauty',
    'Books': 'Books',
    'Cell_Phones_and_Accessories': 'Cell Phones',
    'Clothing_Shoes_and_Jewelry': 'Clothing',
    'Electronics': 'Electronics',
    'Health_and_Household': 'Health',
    'Home_and_Kitchen': 'Home & Kitchen',
    'Kindle_Store': 'Kindle',
    'Tools_and_Home_Improvement': 'Tools'
}

order = list(short_names.keys())
df_ordered = loc_df.set_index('category').loc[order]

print('=== Per-Class LOCO F1 (for comparison notebook) ===')
print(f'f1h_loc_tfidf = {[round(x, 4) for x in df_ordered["f1_human"].values]}')
print(f'f1a_loc_tfidf = {[round(x, 4) for x in df_ordered["f1_ai"].values]}')
print(f'tfidf_loco_f1 = {[round(x, 4) for x in df_ordered["macro_f1"].values]}')
print()
print(f'Human Prec avg: {loc_df["precision_human"].mean():.4f}')
print(f'Human Rec avg: {loc_df["recall_human"].mean():.4f}')
print(f'Human F1 avg: {loc_df["f1_human"].mean():.4f}')
print(f'AI Prec avg: {loc_df["precision_ai"].mean():.4f}')
print(f'AI Rec avg: {loc_df["recall_ai"].mean():.4f}')
print(f'AI F1 avg: {loc_df["f1_ai"].mean():.4f}')
print(f'Macro-F1 avg: {loc_df["macro_f1"].mean():.4f}')
print(f'Accuracy avg: {loc_df["accuracy"].mean():.4f}')

=== Per-Class LOCO F1 (for comparison notebook) ===
f1h_loc_tfidf = [np.float64(0.9231), np.float64(0.9174), np.float64(0.9412), np.float64(0.873), np.float64(0.8092), np.float64(0.9384), np.float64(0.9262), np.float64(0.8772), np.float64(0.7755), np.float64(0.9138)]
f1a_loc_tfidf = [np.float64(0.8519), np.float64(0.9032), np.float64(0.9545), np.float64(0.8049), np.float64(0.7986), np.float64(0.9395), np.float64(0.8817), np.float64(0.8511), np.float64(0.9106), np.float64(0.9038)]
tfidf_loco_f1 = [np.float64(0.8875), np.float64(0.9103), np.float64(0.9479), np.float64(0.8389), np.float64(0.8039), np.float64(0.939), np.float64(0.9039), np.float64(0.8641), np.float64(0.843), np.float64(0.9088)]

Human Prec avg: 0.8583
Human Rec avg: 0.9271
Human F1 avg: 0.8895
AI Prec avg: 0.9204
AI Rec avg: 0.8477
AI F1 avg: 0.8800
Macro-F1 avg: 0.8847
Accuracy avg: 0.8903
